# 05 - Silver Validation
### Goal
- validate all silver tables
- check nulls, negative values, schema consistency, key field quality

In [0]:
%run ../01_setup/00_config

In [0]:
from utils.validation_utils import get_logger, run_table_validation, display_summary

log = get_logger()

## Table validation

In [0]:
expected_schemas = {
    silver_market_prices_table: {
        "required_columns": {
            "event_date", "region", "market_type",
            "price_eur_mwh", "volume_mwh", "source_system",
            "last_updated_ts", "is_high_price", "price_band"
        },
        "key_columns":    ["event_date", "region", "price_eur_mwh"],
        "numeric_checks": [("price_eur_mwh", 0), ("volume_mwh", 0)]
    },
    silver_weather_table: {
        "required_columns": {
            "event_date", "region", "temperature_c",
            "wind_speed_kmh", "precipitation_mm",
            "weather_alert_level", "source_system",
            "last_updated_ts", "is_severe_weather", "wind_class"
        },
        "key_columns":    ["event_date", "region"],
        "numeric_checks": [("wind_speed_kmh", 0), ("precipitation_mm", 0)]
    },
    silver_grid_events_table: {
        "required_columns": {
            "event_id", "event_date", "region", "asset_id",
            "event_type", "severity", "duration_minutes",
            "source_system", "last_updated_ts",
            "is_critical_event", "duration_band"
        },
        "key_columns":    ["event_id", "event_date", "region"],
        "numeric_checks": [("duration_minutes", 0)]
    },
    silver_integrated_table: {
        "required_columns": {
            "event_id", "event_date", "region", "asset_id",
            "event_type", "severity", "duration_minutes",
            "is_critical_event", "duration_band",
            "avg_price_eur_mwh", "total_volume_mwh",
            "avg_temperature_c", "avg_wind_speed_kmh",
            "total_precipitation_mm", "has_severe_weather",
            "wind_class", "country_code", "operating_zone"
        },
        "key_columns":    ["event_id", "event_date", "region"],
        "numeric_checks": [
            ("avg_price_eur_mwh", 0),
            ("avg_wind_speed_kmh", 0),
            ("duration_minutes", 0)
        ]
    }
}

results, validation_passed = run_table_validation(spark, expected_schemas, log)

## Validation summary

In [0]:
display_summary(spark, results, validation_passed, "Silver")